# Chapter 11 — Memory Coalescing & Vectorized Loads

> Course: **llm.c — Zero to Hero**, Chapter 11 of ~20.
> Builds on Chapter 10 (grid-stride loops).

GELU was bandwidth-bound. We're hitting ~625 GB/s on the RTX 4080 from Chapter 9-10 — solid, but not peak. The 4080 has ~720 GB/s of theoretical bandwidth, and tuned kernels can hit ~95% of that. This chapter is about how to close the gap.

The two ideas:

1. [**Memory coalescing**](https://developer.nvidia.com/blog/unlock-gpu-performance-global-memory-access-in-cuda/) — when 32 threads in a warp issue loads, the hardware *combines* them into one transaction *if and only if* their addresses are contiguous. Non-contiguous = 32 transactions. We've been getting this for free; now we'll see what breaks it.
2. [**Vectorized loads**](https://developer.nvidia.com/blog/cuda-pro-tip-increase-performance-with-vectorized-memory-access/) — we can issue a single 128-bit memory transaction that loads 4 floats (or 8 bf16s) at once, instead of four 32-bit ones. This is the `Packed128` (`x128`) type used throughout `llm.c`'s production kernels.

### Learning objectives

By the end of this chapter you will:

- Explain *coalesced* vs *strided* memory access patterns at the warp level.
- Use `float4` for 128-bit vectorized loads/stores.
- Read `dev/cuda/gelu_forward.cu`'s **kernel 2** with the `x128` Packed128 type, and explain how it raises arithmetic intensity.
- Beat the kernel-1 GELU bandwidth on the same GPU.


## 1. Concept — How a Warp Issues Loads

A **warp** is 32 threads executing in lockstep. When they hit a memory load:

```c
float v = inp[i];
```

…the hardware looks at all 32 thread's addresses *together*. If they hit 32 *contiguous* floats (128 contiguous bytes — exactly one cache line), the load **coalesces** into **one** memory transaction. If addresses are scattered, you can pay up to **32 transactions** for the same 32 values.

The pattern that gives perfect coalescing is the one we've been writing without thinking about it:

```c
int i = blockIdx.x * blockDim.x + threadIdx.x;
out[i] = ...; inp[i] = ...;            // each thread → adjacent memory address
```

Within one warp, `threadIdx.x` runs `0..31` (or some 32-thread sub-range), so `i` runs across 32 adjacent indices. **One cache line, one transaction.** This is why kernels 1 from Chapter 9-10 reach hundreds of GB/s without us doing anything.

The pattern that **breaks** coalescing:

```c
out[i * stride] = ...;                  // strided: thread 0 hits index 0, thread 1 hits index `stride`, ...
```

Now 32 threads hit 32 different cache lines. **32 transactions for one warp's worth of work.**

### Concrete demo: contiguous vs strided

Let's measure both. We'll write two kernels that read `N` floats from a buffer and sum them — one contiguous, one strided — and time them.


In [ ]:
!mkdir -p course/ch11_build


In [ ]:
%%writefile course/ch11_build/coalesce_demo.cu
#include <stdio.h>
#include <cuda_runtime.h>

// Each thread reads ONE element only, and writes to a small dense output buffer.
// Two kernels differ in *which* element each thread reads.

// Coalesced: thread tid reads inp[tid] — adjacent threads → adjacent addresses → 1 cache line per warp.
__global__ void read_coalesced(float* out, const float* inp, int n_outputs) {
    int tid = blockIdx.x * blockDim.x + threadIdx.x;
    if (tid < n_outputs) out[tid] = inp[tid];
}

// Strided: thread tid reads inp[tid * STRIDE] — adjacent threads STRIDE apart → 32 cache lines per warp.
// We deliberately fetch MUCH more memory than we use: each warp pulls 32 cache lines (4 KB)
// to consume just 32 floats (128 bytes). This is the "non-coalesced" worst case.
__global__ void read_strided(float* out, const float* inp, int n_outputs, int STRIDE) {
    int tid = blockIdx.x * blockDim.x + threadIdx.x;
    if (tid < n_outputs) out[tid] = inp[tid * STRIDE];
}

int main(void) {
    // Big enough to thrash L2: ~256 MB input, 32 MB output.
    const int STRIDE    = 32;
    const int n_outputs = 8 * 1024 * 1024;
    const int N         = n_outputs * STRIDE;
    float *d_in, *d_out;
    cudaMalloc(&d_in,  (size_t)N*4); cudaMalloc(&d_out, (size_t)n_outputs*4);
    cudaEvent_t s, e; cudaEventCreate(&s); cudaEventCreate(&e);
    int iters = 30;
    int block = 256;
    int grid  = (n_outputs + block - 1) / block;

    // warmup + time coalesced
    read_coalesced<<<grid, block>>>(d_out, d_in, n_outputs); cudaDeviceSynchronize();
    cudaEventRecord(s);
    for (int k = 0; k < iters; k++) read_coalesced<<<grid, block>>>(d_out, d_in, n_outputs);
    cudaEventRecord(e); cudaEventSynchronize(e);
    float ms_coal; cudaEventElapsedTime(&ms_coal, s, e);

    // time strided
    read_strided<<<grid, block>>>(d_out, d_in, n_outputs, STRIDE); cudaDeviceSynchronize();
    cudaEventRecord(s);
    for (int k = 0; k < iters; k++) read_strided<<<grid, block>>>(d_out, d_in, n_outputs, STRIDE);
    cudaEventRecord(e); cudaEventSynchronize(e);
    float ms_str; cudaEventElapsedTime(&ms_str, s, e);

    // "Useful bytes" = output size, regardless of how much was fetched
    float useful_bytes = (float)n_outputs * 4.0f * 2.0f;     // read+write both float
    auto bw = [useful_bytes](float ms_per_iter){
        return (useful_bytes / 1e9f) / (ms_per_iter / 1000.0f);
    };
    printf("n_outputs=%d  STRIDE=%d  input=%.0f MB\n",
           n_outputs, STRIDE, (float)N*4.0f/1024.0f/1024.0f);
    printf("read coalesced : %6.3f ms/iter   %6.1f GB/s (effective)\n",
           ms_coal/iters, bw(ms_coal/iters));
    printf("read strided   : %6.3f ms/iter   %6.1f GB/s (effective)\n",
           ms_str/iters,  bw(ms_str/iters));
    printf("strided is %.2fx slower\n", ms_str / ms_coal);

    cudaFree(d_in); cudaFree(d_out);
    return 0;
}


In [ ]:
!nvcc -O2 -o course/ch11_build/coalesce_demo course/ch11_build/coalesce_demo.cu && ./course/ch11_build/coalesce_demo


You should see the strided kernel achieve markedly lower effective bandwidth than the coalesced version. On an RTX 4080 SUPER this benchmark runs the strided kernel **~14× slower** — the input is deliberately sized at 1 GB so the warp's 32 scattered cache lines overflow the 64 MB L2 and fall through to DRAM. Shrink the working set and the L2 would absorb most of the penalty and the gap would close; widen it (real-world large tensors) and it stays large. Either way the ordering never reverses.

You may also notice the *coalesced* number (~2000 GB/s here) sails **past** the card's ~720 GB/s DRAM bandwidth. That's not magic: the coalesced kernel's ~32 MB working set fits inside the 64 MB L2, so it's served from cache. The strided kernel's 1 GB span can't fit, so it pays full DRAM cost *plus* the wasted transactions — which is exactly the gap you're measuring.

The takeaway is unconditional: **make consecutive threads touch consecutive addresses.** This is rule #1 of CUDA performance.


## 2. Concept — Vectorized Loads with `float4`

Even when accesses are coalesced, you can do better. A single thread can issue **a single 128-bit memory transaction** that loads 4 floats (or 2 doubles, or 8 bf16) at once, by using a vector type:

```c
float4 v = ((float4*)inp)[i];     // loads inp[4*i .. 4*i+3] as one 128-bit transaction
v.x = relu(v.x); v.y = relu(v.y); v.z = relu(v.z); v.w = relu(v.w);
((float4*)out)[i] = v;
```

This generates one `LDG.E.128` (or `LD.128`) instruction per element-of-4 instead of four `LDG.E.32`s. Fewer instructions, fewer transactions, more bandwidth.

`llm.c`'s `Packed128` type (in `llmc/cuda_utils.cuh`) generalizes this: it's a 128-bit container that holds 4 floats or 8 bf16s depending on `floatX`, with helpers `load128`, `store128`, `load128cs` (cache-streaming for write-only data), etc. We'll see the real one in Chapter 17 (mixed precision); for now, plain `float4` makes the point.


## 3. Demo — GELU with `float4` Vectorized Loads

Here's `gelu_forward_kernel2`-style code, simplified to use `float4` directly:


In [ ]:
%%writefile course/ch11_build/gelu_vec.cu
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <cuda_runtime.h>

#define GELU_SCALING_FACTOR sqrtf(2.0f / M_PI)

// kernel 1: scalar, one float per thread (Chapter 9-10 baseline)
__global__ void gelu_kernel1(float* out, const float* inp, int N) {
    int tid    = blockIdx.x * blockDim.x + threadIdx.x;
    int stride = blockDim.x * gridDim.x;
    for (int i = tid; i < N; i += stride) {
        float x = inp[i];
        float cube = 0.044715f * x * x * x;
        out[i] = 0.5f * x * (1.0f + tanhf(GELU_SCALING_FACTOR * (x + cube)));
    }
}

// kernel 2: float4 vectorized, four floats per thread
__device__ __forceinline__ float gelu_scalar(float x) {
    float cube = 0.044715f * x * x * x;
    return 0.5f * x * (1.0f + tanhf(GELU_SCALING_FACTOR * (x + cube)));
}

__global__ void gelu_kernel2(float* out, const float* inp, int N) {
    // each thread handles 4 floats; index in float4 units
    int tid    = blockIdx.x * blockDim.x + threadIdx.x;
    int stride = blockDim.x * gridDim.x;
    int N4     = N / 4;
    for (int i = tid; i < N4; i += stride) {
        float4 v = ((float4*)inp)[i];                 // 1 LDG.128
        float4 r;
        r.x = gelu_scalar(v.x);
        r.y = gelu_scalar(v.y);
        r.z = gelu_scalar(v.z);
        r.w = gelu_scalar(v.w);
        ((float4*)out)[i] = r;                        // 1 STG.128
    }
    // (assumes N % 4 == 0, which it does for any GPT-2 hidden size)
}

int main(void) {
    const int N = 1 << 24;
    float *d_in, *d_out, *d_out2;
    cudaMalloc(&d_in,  N*4); cudaMalloc(&d_out, N*4); cudaMalloc(&d_out2, N*4);

    // populate input with arbitrary values (the kernels just need to do the same math)
    cudaMemset(d_in, 0x3f, N*4);  // bit pattern → some float

    cudaEvent_t s, e; cudaEventCreate(&s); cudaEventCreate(&e);
    int iters = 50, block = 256, grid = 4096;

    // kernel 1
    gelu_kernel1<<<grid, block>>>(d_out, d_in, N); cudaDeviceSynchronize();
    cudaEventRecord(s);
    for (int k = 0; k < iters; k++) gelu_kernel1<<<grid, block>>>(d_out, d_in, N);
    cudaEventRecord(e); cudaEventSynchronize(e);
    float ms1; cudaEventElapsedTime(&ms1, s, e);

    // kernel 2 (float4)
    gelu_kernel2<<<grid, block>>>(d_out2, d_in, N); cudaDeviceSynchronize();
    cudaEventRecord(s);
    for (int k = 0; k < iters; k++) gelu_kernel2<<<grid, block>>>(d_out2, d_in, N);
    cudaEventRecord(e); cudaEventSynchronize(e);
    float ms2; cudaEventElapsedTime(&ms2, s, e);

    // correctness: compare a sample of outputs
    float a, b;
    cudaMemcpy(&a, d_out, 4, cudaMemcpyDeviceToHost);
    cudaMemcpy(&b, d_out2, 4, cudaMemcpyDeviceToHost);
    auto bw = [](float ms_per_iter, int N){
        return (2.0f*N*4.0f/1e9f) / (ms_per_iter/1000.0f);
    };
    printf("kernel 1 (scalar) : %6.3f ms/iter   %6.1f GB/s\n", ms1/iters, bw(ms1/iters, N));
    printf("kernel 2 (float4) : %6.3f ms/iter   %6.1f GB/s\n", ms2/iters, bw(ms2/iters, N));
    printf("speedup           : %.2fx\n", ms1/ms2);
    printf("sample agreement  : %.2e\n", (a == b) ? 0.0 : (double)(a - b));

    cudaFree(d_in); cudaFree(d_out); cudaFree(d_out2);
    return 0;
}


In [ ]:
!nvcc -O2 -o course/ch11_build/gelu_vec course/ch11_build/gelu_vec.cu && ./course/ch11_build/gelu_vec


When the scalar version is already saturating the memory bus (as on a fast GPU with a large L2), the `float4` win can be modest — a few percent. On more constrained GPUs or kernels where the scalar version isn't already bandwidth-limited, the gap is bigger (often 1.5–2×). The `float4` version is doing the same compute (4 GELUs), so any gain is *purely* a memory-system win — fewer transactions, fewer instruction-issue cycles per byte transferred.

The bigger payoff comes with **bf16**: now `Packed128` packs *8* values per 128-bit transaction. The factor-of-8 reduction in transactions can absolutely move the needle even on a 4080.


## 4. The `Packed128` / `x128` Type in `llm.c`

The repo's `llmc/cuda_utils.cuh` defines a templated `Packed128` that's the production version of what we just did:

```cpp
template<class ElementType>
struct alignas(16) Packed128 {
    static constexpr const size_t size = 16 / sizeof(ElementType);
    ElementType payload[size];
};

typedef Packed128<floatX> x128;       // 4 floats if FP32, 8 bf16s if BF16

template<class ElementType>
__device__ Packed128<ElementType> load128(const ElementType* address);

template<class ElementType>
__device__ Packed128<ElementType> load128cs(const ElementType* address);   // cache-streaming

template<class ElementType>
__device__ void store128(ElementType* target, Packed128<ElementType> value);
```

Production kernels in `llmc/encoder.cuh`, `llmc/layernorm.cuh`, `llmc/matmul.cuh`, etc. use `load128` and `store128` everywhere. Since `floatX` is `nv_bfloat16` in mixed-precision builds, **one 128-bit transaction loads 8 bf16 elements per thread** — exactly the right granularity for the GPU's memory bus.

We'll meet this type in earnest in Chapter 17 when we discuss mixed precision. For this chapter, just internalize: **8 bf16 (or 4 fp32) per thread, one 128-bit transaction.** That's the production memory pattern.


## 5. Translation Bridge

| Pattern | What it does | When to use |
|---|---|---|
| `out[i] = ...` with `i = tid` | one float per thread, coalesced | small / unaligned data, bf16 isn't needed |
| `((float4*)out)[i]` | 4 floats per thread, 1 transaction | fp32, `N % 4 == 0` |
| `x128 = Packed128<floatX>` | 4 fp32 OR 8 bf16 per thread | production `llm.c` kernels |
| `load128cs` instead of `load128` | bypass cache (write-only data) | streaming through giant tensors |

Three rules to take with you for every CUDA kernel:

1. **Adjacent threads → adjacent memory.** Always.
2. **Use the widest load you can** (`float4` or `Packed128`) when alignment allows.
3. **Profile.** If your kernel isn't at ~80% peak bandwidth and you're bandwidth-bound, you're missing one of the above.


## 6. TODO Exercise — `float4` Residual

In [ ]:
%%writefile course/ch11_build/exercise1.cu
#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>

// TODO: write a residual_forward kernel that uses float4 to do 4 adds per thread.
__global__ void residual_vec(float* out, const float* a, const float* b, int N) {
    int tid = blockIdx.x * blockDim.x + threadIdx.x;
    int stride = blockDim.x * gridDim.x;
    int N4 = N / 4;
    for (int i = tid; i < N4; i += stride) {
        // TODO: load float4 va = ((float4*)a)[i]; vb = ((float4*)b)[i];
        // TODO: compute va + vb componentwise into a float4 r
        // TODO: store ((float4*)out)[i] = r;
    }
}

int main(void) {
    const int N = 1 << 20;
    float *h_a = (float*) malloc(N*4);
    float *h_b = (float*) malloc(N*4);
    float *h_o = (float*) malloc(N*4);
    for (int i = 0; i < N; i++) { h_a[i] = (float)i; h_b[i] = (float)(2*i); }
    float *d_a, *d_b, *d_o;
    cudaMalloc(&d_a, N*4); cudaMalloc(&d_b, N*4); cudaMalloc(&d_o, N*4);
    cudaMemcpy(d_a, h_a, N*4, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, h_b, N*4, cudaMemcpyHostToDevice);
    residual_vec<<<256, 256>>>(d_o, d_a, d_b, N);
    cudaMemcpy(h_o, d_o, N*4, cudaMemcpyDeviceToHost);
    int correct = 1;
    for (int i = 0; i < N; i++) if (h_o[i] != 3.0f*i) { correct = 0; break; }
    printf("%s  (h_o[0..3] = %.0f %.0f %.0f %.0f)\n", correct ? "PASS" : "FAIL",
           h_o[0], h_o[1], h_o[2], h_o[3]);
    cudaFree(d_a); cudaFree(d_b); cudaFree(d_o);
    free(h_a); free(h_b); free(h_o);
    return 0;
}


In [ ]:
!nvcc -O2 -o course/ch11_build/exercise1 course/ch11_build/exercise1.cu && ./course/ch11_build/exercise1


### Solution

In [ ]:
%%writefile course/ch11_build/exercise1_sol.cu
#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>

__global__ void residual_vec(float* out, const float* a, const float* b, int N) {
    int tid = blockIdx.x * blockDim.x + threadIdx.x;
    int stride = blockDim.x * gridDim.x;
    int N4 = N / 4;
    for (int i = tid; i < N4; i += stride) {
        float4 va = ((const float4*)a)[i];
        float4 vb = ((const float4*)b)[i];
        float4 r;
        r.x = va.x + vb.x; r.y = va.y + vb.y;
        r.z = va.z + vb.z; r.w = va.w + vb.w;
        ((float4*)out)[i] = r;
    }
}

int main(void) {
    const int N = 1 << 20;
    float *h_a = (float*) malloc(N*4);
    float *h_b = (float*) malloc(N*4);
    float *h_o = (float*) malloc(N*4);
    for (int i = 0; i < N; i++) { h_a[i] = (float)i; h_b[i] = (float)(2*i); }
    float *d_a, *d_b, *d_o;
    cudaMalloc(&d_a, N*4); cudaMalloc(&d_b, N*4); cudaMalloc(&d_o, N*4);
    cudaMemcpy(d_a, h_a, N*4, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, h_b, N*4, cudaMemcpyHostToDevice);
    residual_vec<<<256, 256>>>(d_o, d_a, d_b, N);
    cudaMemcpy(h_o, d_o, N*4, cudaMemcpyDeviceToHost);
    int correct = 1;
    for (int i = 0; i < N; i++) if (h_o[i] != 3.0f*i) { correct = 0; break; }
    printf("%s  (h_o[0..3] = %.0f %.0f %.0f %.0f)\n", correct ? "PASS" : "FAIL",
           h_o[0], h_o[1], h_o[2], h_o[3]);
    cudaFree(d_a); cudaFree(d_b); cudaFree(d_o);
    free(h_a); free(h_b); free(h_o);
    return 0;
}


In [ ]:
!nvcc -O2 -o course/ch11_build/exercise1_sol course/ch11_build/exercise1_sol.cu && ./course/ch11_build/exercise1_sol


## Further Reading

**Source of truth**

- [_How to Access Global Memory Efficiently in CUDA C/C++ Kernels_](https://developer.nvidia.com/blog/how-access-global-memory-efficiently-cuda-c-kernels/) (Mark Harris) — the canonical explanation of warp-level coalescing, aligned segments, and why strided access multiplies transactions.
- [_CUDA Pro Tip: Increase Performance with Vectorized Memory Access_](https://developer.nvidia.com/blog/cuda-pro-tip-increase-performance-with-vectorized-memory-access/) — `float2`/`float4` loads, alignment requirements, and the instruction-count win behind `Packed128`.
- `dev/cuda/gelu_forward.cu` (kernel 2) and `llmc/cuda_utils.cuh` (`Packed128`/`x128`, `load128`/`store128`) in this repo — the production code this chapter models.

**Going deeper**

- [CUDA C++ Best Practices Guide — Coalesced Access to Global Memory](https://docs.nvidia.com/cuda/cuda-c-best-practices-guide/index.html) — the official optimization reference.
- [_An Efficient Matrix Transpose in CUDA C/C++_](https://developer.nvidia.com/blog/efficient-matrix-transpose-cuda-cc/) — a worked example of fixing a non-coalesced access pattern with shared memory (a preview of Chapter 13).


## Recap

You now know:

- **Coalescing**: 32 threads in a warp issuing loads to 32 contiguous addresses → 1 transaction. Strided access → up to 32 transactions. This is the #1 CUDA performance rule.
- **`float4`** issues one 128-bit memory transaction for 4 fp32 values per thread.
- **`Packed128` / `x128`** in `llm.c` is the same idea generalized to bf16/fp16 — 8 bf16 per thread per transaction.

### What's next

**Chapter 12 — Reductions I: Warp-level Primitives.** GELU was element-wise. Softmax and LayerNorm need to compute per-row max and sum — *reductions*. We'll meet `__shfl_down_sync` (warp shuffles) and write a per-warp reduction in 5 lines. Setting up Chapter 13 (block-level reductions with shared memory).

When you're ready, say **"proceed to Chapter 12"**.
